# 5 - Statistical Validation

## Sections
1. Setup & Data Loading
2. T-test on Alpha (H0: mean return = 0)
3. Bootstrap Confidence Intervals for Sharpe Ratio
4. Hansen SPA Test (Multiple Testing Correction)
5. Factor Benchmark Regression
6. Regime Analysis by Realized Volatility
7. Exit Type & Direction Breakdown
8. Summary Table

---
## 1) Setup & Data Loading

In [ ]:
import sys
sys.path.insert(0, '.')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_config, load_btc_5min, load_btc_daily, split_is_oos
from src.backtest_engine import run_backtest, compute_stats
from src.statistics import (
    ttest_alpha, bootstrap_sharpe_ci, hansen_spa_test,
    compute_factor_returns, factor_regression,
    regime_analysis, exit_type_breakdown, direction_breakdown
)

config = load_config()
df_signals = pd.read_csv('Data/df_signals_v2.csv', parse_dates=['Date', 'Expiry'])
df_5m = load_btc_5min(config)
df_btc_daily = load_btc_daily(config)

# Split IS/OOS
df_is, df_oos = split_is_oos(df_signals, config, date_col='Date')
print(f'IS signals: {len(df_is)}, OOS signals: {len(df_oos)}')
print(f'DTE median: {df_signals["DTE"].median():.1f} days')

In [ ]:
# Run backtest with OPTIMAL parameters from sweep (best robust combo N>=50)
GAP = 0.02
SL_MULT = 2.0
SPREAD = 0.014
HP = 14

# IS trades
trades_is = run_backtest(df_is, df_5m, config,
                         gap_threshold=GAP, sl_multiplier=SL_MULT,
                         spread_tolerance=SPREAD, holding_period=HP)
stats_is = compute_stats(trades_is)

# OOS trades
trades_oos = run_backtest(df_oos, df_5m, config,
                          gap_threshold=GAP, sl_multiplier=SL_MULT,
                          spread_tolerance=SPREAD, holding_period=HP)
stats_oos = compute_stats(trades_oos)

print(f'Optimal params: gap={GAP}, sl={SL_MULT}, spread={SPREAD}, HP={HP}d')
print(f'IS:  N={stats_is["N_Trades"]}, Sharpe={stats_is["Sharpe"]:.2f}, '
      f'WR={stats_is["Win_Rate_Pct"]:.1f}%, Return={stats_is["Total_Return_Pct"]:.1f}%, MaxDD={stats_is["Max_Drawdown"]:.1f}%')
print(f'OOS: N={stats_oos["N_Trades"]}, Sharpe={stats_oos["Sharpe"]:.2f}, '
      f'WR={stats_oos["Win_Rate_Pct"]:.1f}%, Return={stats_oos["Total_Return_Pct"]:.1f}%, MaxDD={stats_oos["Max_Drawdown"]:.1f}%')

---
## 2) T-test on Alpha

In [ ]:
returns_is = trades_is['Return_Pct'].values / 100
returns_oos = trades_oos['Return_Pct'].values / 100

print('=== T-test on Alpha (H0: mean return = 0) ===')
for label, ret in [('IS', returns_is), ('OOS', returns_oos)]:
    result = ttest_alpha(ret)
    sig = 'YES' if result['significant_5pct'] else 'NO'
    print(f'{label}: mean={result["mean"]*100:.3f}%, t={result["t_stat"]:.3f}, '
          f'p={result["p_value"]:.4f}, significant={sig} (n={result["n_obs"]})')

---
## 3) Bootstrap Confidence Intervals for Sharpe Ratio

In [ ]:
print('=== Bootstrap 95% CI for Sharpe Ratio ===')
for label, ret in [('IS', returns_is), ('OOS', returns_oos)]:
    boot = bootstrap_sharpe_ci(ret, n_bootstrap=config['statistics']['bootstrap_n_samples'])
    print(f'{label}: Sharpe={boot["sharpe"]:.3f}, 95% CI=[{boot["ci_lower"]:.3f}, '
          f'{boot["ci_upper"]:.3f}], SE={boot["se"]:.3f}')

# Plot bootstrap distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, ret) in zip(axes, [('IS', returns_is), ('OOS', returns_oos)]):
    rng = np.random.RandomState(42)
    n = len(ret)
    sharpes = []
    for _ in range(10000):
        s = ret[rng.randint(0, n, size=n)]
        sharpes.append(s.mean() / s.std() if s.std() > 0 else 0)
    ax.hist(sharpes, bins=50, alpha=0.7, color='steelblue', edgecolor='white')
    boot = bootstrap_sharpe_ci(ret)
    ax.axvline(boot['ci_lower'], color='red', linestyle='--', label=f'95% CI')
    ax.axvline(boot['ci_upper'], color='red', linestyle='--')
    ax.axvline(0, color='black', linestyle='-', alpha=0.5, label='Zero')
    ax.set_title(f'{label} Bootstrap Sharpe Distribution')
    ax.set_xlabel('Sharpe Ratio (per trade)')
    ax.legend()
plt.tight_layout()
plt.show()

---
## 4) Hansen SPA Test (Multiple Testing Correction)

In [ ]:
# Load sweep results if available
import os
if os.path.exists('Data/sweep_IS_v2.csv'):
    df_sweep = pd.read_csv('Data/sweep_IS_v2.csv')
    print(f'Loaded {len(df_sweep)} parameter combinations from sweep')
    
    # Collect returns for each combo (use top 50 by Sharpe)
    top_combos = df_sweep.head(50)
    model_returns_list = []
    for _, row in top_combos.iterrows():
        trades = run_backtest(df_is, df_5m, config,
                              gap_threshold=row['gap_threshold'],
                              sl_multiplier=row['sl_multiplier'],
                              spread_tolerance=row['spread_tolerance'],
                              holding_period=int(row['holding_period']))
        if not trades.empty:
            model_returns_list.append(trades['Return_Pct'].values / 100)
    
    # Benchmark: zero returns
    max_len = max(len(r) for r in model_returns_list)
    base_returns = np.zeros(max_len)
    
    spa_result = hansen_spa_test(base_returns, model_returns_list)
    print(f'\nHansen SPA Test:')
    print(f'  H0: No model beats zero benchmark')
    print(f'  Max t-stat: {spa_result["max_t_stat"]:.3f}')
    print(f'  p-value: {spa_result["p_value"]:.4f}')
    print(f'  N models tested: {spa_result["n_models"]}')
    print(f'  Conclusion: {"Reject H0 - at least one model has genuine alpha" if spa_result["p_value"] < 0.05 else "Cannot reject H0"}')
else:
    print('Run parameter sweep first (sweep_IS_v2.csv not found)')

---
## 5) Factor Benchmark Regression

In [ ]:
print('=== Factor Regression (IS) ===')
trades_is_factors = compute_factor_returns(trades_is, df_btc_daily, df_signals)
fr_is = factor_regression(trades_is_factors)
print(f'Alpha: {fr_is["alpha"]*100:.3f}% per trade')
print(f'Alpha t-stat: {fr_is["alpha_tstat"]:.3f}')
print(f'Alpha p-value: {fr_is["alpha_pvalue"]:.4f}')
print(f'R-squared: {fr_is["r_squared"]:.4f}')
print(f'\nFactor loadings:')
for factor, vals in fr_is.get('factor_betas', {}).items():
    print(f'  {factor}: beta={vals["beta"]:.4f} (se={vals["se"]:.4f})')

print('\n=== Factor Regression (OOS) ===')
trades_oos_factors = compute_factor_returns(trades_oos, df_btc_daily, df_signals)
fr_oos = factor_regression(trades_oos_factors)
print(f'Alpha: {fr_oos["alpha"]*100:.3f}% per trade')
print(f'Alpha t-stat: {fr_oos["alpha_tstat"]:.3f}')
print(f'Alpha p-value: {fr_oos["alpha_pvalue"]:.4f}')
print(f'R-squared: {fr_oos["r_squared"]:.4f}')

---
## 6) Regime Analysis

In [ ]:
print('=== Regime Analysis (by Realized Volatility) ===')
print(f'Thresholds: Low < {config["statistics"]["regime_thresholds"]["low_vol"]}%, '
      f'Medium {config["statistics"]["regime_thresholds"]["low_vol"]}-{config["statistics"]["regime_thresholds"]["high_vol"]}%, '
      f'High > {config["statistics"]["regime_thresholds"]["high_vol"]}%')

for label, trades_f in [('IS', trades_is_factors), ('OOS', trades_oos_factors)]:
    ra = regime_analysis(trades_f, config)
    if not ra.empty:
        print(f'\n{label}:')
        print(ra.to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ra_is = regime_analysis(trades_is_factors, config)
if not ra_is.empty:
    colors = {'Low': '#2196F3', 'Medium': '#FF9800', 'High': '#F44336'}
    ax = axes[0]
    ax.bar(ra_is['regime'], ra_is['sharpe'], color=[colors.get(r, 'gray') for r in ra_is['regime']])
    ax.set_title('Sharpe by Regime (IS)')
    ax.set_ylabel('Sharpe Ratio')
    ax.axhline(y=0, color='black', linewidth=0.5)
    
    ax = axes[1]
    ax.bar(ra_is['regime'], ra_is['win_rate'], color=[colors.get(r, 'gray') for r in ra_is['regime']])
    ax.set_title('Win Rate by Regime (IS)')
    ax.set_ylabel('Win Rate (%)')
    ax.axhline(y=50, color='black', linewidth=0.5, linestyle='--')
    
    ax = axes[2]
    ax.bar(ra_is['regime'], ra_is['tp_rate'], color=[colors.get(r, 'gray') for r in ra_is['regime']])
    ax.set_title('TP Rate by Regime (IS)')
    ax.set_ylabel('TP Rate (%)')

plt.tight_layout()
plt.show()

---
## 7) Exit Type & Direction Breakdown

In [ ]:
for label, trades in [('IS', trades_is), ('OOS', trades_oos)]:
    print(f'\n=== {label} Exit Type Breakdown ===')
    etb = exit_type_breakdown(trades)
    if not etb.empty:
        print(etb.to_string(index=False))
    
    print(f'\n=== {label} Direction Breakdown ===')
    db = direction_breakdown(trades)
    if not db.empty:
        print(db.to_string(index=False))

# Plot exit type pie charts
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (label, trades) in zip(axes, [('IS', trades_is), ('OOS', trades_oos)]):
    if 'Exit_Type' in trades.columns and not trades.empty:
        counts = trades['Exit_Type'].value_counts()
        ax.pie(counts.values, labels=counts.index, autopct='%1.1f%%',
               colors=['#4CAF50', '#F44336', '#FF9800', '#2196F3'])
        ax.set_title(f'{label} Exit Type Distribution')
plt.tight_layout()
plt.show()

---
## 8) Summary Table

In [ ]:
summary = pd.DataFrame([
    {
        'Metric': 'N Trades',
        'IS': stats_is['N_Trades'],
        'OOS': stats_oos['N_Trades'],
    },
    {
        'Metric': 'Sharpe Ratio',
        'IS': f"{stats_is['Sharpe']:.2f}",
        'OOS': f"{stats_oos['Sharpe']:.2f}",
    },
    {
        'Metric': 'Win Rate (%)',
        'IS': f"{stats_is['Win_Rate_Pct']:.1f}",
        'OOS': f"{stats_oos['Win_Rate_Pct']:.1f}",
    },
    {
        'Metric': 'TP Rate (%)',
        'IS': f"{stats_is['TP_Rate']:.1f}",
        'OOS': f"{stats_oos['TP_Rate']:.1f}",
    },
    {
        'Metric': 'Total Return (%)',
        'IS': f"{stats_is['Total_Return_Pct']:.1f}",
        'OOS': f"{stats_oos['Total_Return_Pct']:.1f}",
    },
    {
        'Metric': 'Max Drawdown (%)',
        'IS': f"{stats_is['Max_Drawdown']:.1f}",
        'OOS': f"{stats_oos['Max_Drawdown']:.1f}",
    },
    {
        'Metric': 'Alpha (t-stat)',
        'IS': f"{fr_is['alpha_tstat']:.3f} (p={fr_is['alpha_pvalue']:.4f})",
        'OOS': f"{fr_oos['alpha_tstat']:.3f} (p={fr_oos['alpha_pvalue']:.4f})",
    },
    {
        'Metric': 'Sharpe CI (95%)',
        'IS': f"[{bootstrap_sharpe_ci(returns_is)['ci_lower']:.3f}, {bootstrap_sharpe_ci(returns_is)['ci_upper']:.3f}]",
        'OOS': f"[{bootstrap_sharpe_ci(returns_oos)['ci_lower']:.3f}, {bootstrap_sharpe_ci(returns_oos)['ci_upper']:.3f}]",
    },
])

print('=== Summary: IS vs OOS ===')
print(summary.to_string(index=False))